#1.Import libraries

In [9]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

#2: Load and Preprocess the Data

In [10]:
# Set parameters
max_features = 10000  # Only use the top 10,000 most frequent words
maxlen = 200         # Cut reviews after 200 words

print("Loading data...")
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=max_features)

print(f"Training sequences: {len(x_train)}")
print(f"Testing sequences: {len(x_test)}")

# Pad sequences to ensure uniform input length
print("Padding sequences...")
x_train = pad_sequences(x_train, maxlen=maxlen)
x_test = pad_sequences(x_test, maxlen=maxlen)

print(f"x_train shape: {x_train.shape}")
print(f"x_test shape: {x_test.shape}")

Loading data...
Training sequences: 25000
Testing sequences: 25000
Padding sequences...
x_train shape: (25000, 200)
x_test shape: (25000, 200)


#3.Build LSTM architecture

In [11]:
model = Sequential()
# Embedding layer: 10000 words vocabulary, 128 dimensions for the dense vectors
model.add(Embedding(input_dim=max_features, output_dim=128, input_length=maxlen))

# LSTM layer with 64 units
model.add(LSTM(64, dropout=0.2, recurrent_dropout=0.2))

# Output layer for binary classification (Positive/Negative)
model.add(Dense(1, activation='sigmoid'))

# Compile the model
model.compile(loss='binary_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])

model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

#4.Train the model

In [12]:
batch_size = 32
epochs = 10  # Keep epochs low for a quick test, increase for better accuracy

print("Training the model...")
history = model.fit(x_train, y_train,
                    batch_size=batch_size,
                    epochs=epochs,
                    validation_data=(x_test, y_test))

Training the model...
Epoch 1/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 599s 762ms/step - accuracy: 0.7192 - loss: 0.5353 - val_accuracy: 0.7898 - val_loss: 0.4517
Epoch 2/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 599s 766ms/step - accuracy: 0.8591 - loss: 0.3405 - val_accuracy: 0.8580 - val_loss: 0.3553
Epoch 3/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 597s 764ms/step - accuracy: 0.8912 - loss: 0.2771 - val_accuracy: 0.8538 - val_loss: 0.3531
Epoch 4/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 598s 764ms/step - accuracy: 0.9070 - loss: 0.2396 - val_accuracy: 0.7564 - val_loss: 0.5398
Epoch 5/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 598s 765ms/step - accuracy: 0.9199 - loss: 0.2064 - val_accuracy: 0.8660 - val_loss: 0.3703
Epoch 6/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 597s 763ms/step - accuracy: 0.9475 - loss: 0.1414 - val_accuracy: 0.8635 - val_loss: 0.3768
Epoch 7/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 646s 826ms/step - accuracy: 0.9611 - loss: 0.1056 - val_accuracy: 0.8594 - val_loss: 0.4084
Epoch 8/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 593s 759ms/ste

#5.Evaluate

In [13]:
# Evaluate the model
score, acc = model.evaluate(x_test, y_test, batch_size=batch_size)
print(f"Test accuracy: {acc * 100:.2f}%")

# Create a function to predict custom text
def predict_sentiment(text):
    # Get the word index from IMDB
    word_index = imdb.get_word_index()

    # Tokenize and encode the text
    words = text.lower().split()
    encoded_text = []
    for word in words:
        if word in word_index and word_index[word] < max_features:
            # Shift by 3 because 0, 1, and 2 are reserved indices
            encoded_text.append(word_index[word] + 3)
        else:
            encoded_text.append(2) # 2 represents OOV (Out of Vocabulary)

    # Pad the sequence
    padded_text = pad_sequences([encoded_text], maxlen=maxlen)

    # Predict
    prediction = model.predict(padded_text)[0][0]

    if prediction >= 0.5:
        return f"Positive ({prediction:.4f})"
    elif  prediction <=0.4:
      return f"Negative ({prediction:.4f})"
    else:
      return f"Neutral ({prediction:.4f})"


# Try it out!
review1 = "this movie was absolutely fantastic and I loved the acting"
review2 = "terrible plot and awful characters it was a waste of time"


print(f"Review 1 Sentiment: {predict_sentiment(review1)}")
print(f"Review 2 Sentiment: {predict_sentiment(review2)}")


782/782 ━━━━━━━━━━━━━━━━━━━━ 96s 123ms/step - accuracy: 0.8573 - loss: 0.5349
Test accuracy: 85.78%
1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 454ms/step
Review 1 Sentiment: Positive (0.9270)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 149ms/step
Review 2 Sentiment: Negative (0.0012)
